# Train.py – Evaluation Metrics (Accuracy / Precision / Recall / F1)

This notebook explains what this block in `train.py` is doing:

```python
# 5) evaluate
preds = model.predict(X_test)

acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds, zero_division=0)
rec = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)
```

Assumption for phishing detection:
- `label = 1` means **phishing** (positive class)
- `label = 0` means **legit** (negative class)


## 1) What is `preds = model.predict(X_test)`?

- `X_test` = the **test features** (e.g., TF‑IDF vectors) that the model has **not** been trained on.
- `model.predict(X_test)` outputs a prediction for each row in `X_test`.
- For a binary classifier like Logistic Regression, those predictions are usually **0 or 1**.

So `preds` is basically: *“What does the model think each test email is?”*

## 2) Confusion Matrix (the foundation of the metrics)

All the metrics below come from counting these four cases:

- **TP (True Positive):** actual phishing (1), predicted phishing (1)
- **FP (False Positive):** actual legit (0), predicted phishing (1)  
  *(false alarm — legit email flagged as phishing)*
- **TN (True Negative):** actual legit (0), predicted legit (0)
- **FN (False Negative):** actual phishing (1), predicted legit (0)  
  *(miss — phishing email not caught)*


The confusion matrix returned by scikit‑learn is arranged like this:

```
            Pred 0   Pred 1
Actual 0      TN       FP
Actual 1      FN       TP
```

Let’s unpack it into TN/FP/FN/TP:

In [2]:
tn, fp, fn, tp = cm.ravel()
tn, fp, fn, tp

(np.int64(3), np.int64(1), np.int64(2), np.int64(2))

## 3) Accuracy

**Accuracy** = “Out of all emails, what fraction did we get right?”

**Formula:**
- `(TP + TN) / (TP + TN + FP + FN)`

In phishing detection, accuracy can look good even if the model is bad, **if your dataset has lots of legit emails**.
That’s why we also look at precision/recall/F1.

In [ ]:
acc = accuracy_score(y_test, preds)

## 4) Precision

**Precision** = “When the model says *phishing*, how often is it actually phishing?”

**Formula:**
- `TP / (TP + FP)`

High precision means **few false alarms** (few legit emails incorrectly flagged).

### What does `zero_division=0` do?
If your model predicts **no phishing at all**, then `TP + FP = 0` and precision would be a division by zero.
`zero_division=0` tells sklearn: *“in that case, return 0 instead of crashing.”*

In [ ]:
prec = precision_score(y_test, preds, zero_division=0)

## 5) Recall

**Recall** = “Out of all actual phishing emails, how many did we catch?”

**Formula:**
- `TP / (TP + FN)`

High recall means **few misses** (few phishing emails slip through).

`zero_division=0` matters here too if there are no positives in `y_test` (rare, but can happen in tiny samples).

In [ ]:
rec = recall_score(y_test, preds, zero_division=0)

## 6) F1 Score

**F1** balances precision and recall.

**Formula (harmonic mean):**
- `2 * (precision * recall) / (precision + recall)`

Why harmonic mean? It heavily penalizes if one of them is low.
So a model with precision=1.0 but recall=0.1 will still get a low F1.

In [ ]:
f1 = f1_score(y_test, preds, zero_division=0)

## 7) One simple summary

- **Accuracy:** overall correctness
- **Precision:** *when we warn “phishing”, are we usually right?*
- **Recall:** *did we catch most phishing emails?*
- **F1:** one number that balances precision & recall

In phishing detection, many projects care a lot about **recall** (don’t miss phishing),
but you still want decent **precision** (avoid annoying false alarms).